In [0]:
import sys
import mlflow

from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if not (src_path / "energyflow").is_dir():
    raise FileNotFoundError(
        f"EnergyFlow source directory not found: {src_path}"
    )

sys.path.insert(0, str(src_path))

from energyflow.training.splits import (
    prepare_series,
    create_windows,
    split_windows,
)
from energyflow.config import get_table_name

from energyflow.config import load_config
from energyflow.training.run_baselines import run_baselines

In [0]:
config = load_config(project_root / "config" / "project.yaml")

results = run_baselines(spark, config)

display(results)

In [0]:
df = prepare_series(spark, get_table_name(config, "features"))

windows = create_windows(
    df,
    lookback=168,
    horizon=24,
    stride=24,
)

splits = split_windows(
    windows,
    df,
    train_ratio=0.70,
    validation_ratio=0.15,
)

assert len(splits["train"]) > 0
assert len(splits["validation"]) > 0
assert len(splits["test"]) > 0

assert (
    splits["train"]["forecast_end"].max()
    < splits["validation"]["forecast_start"].min()
)

assert (
    splits["validation"]["forecast_end"].max()
    < splits["test"]["forecast_start"].min()
)

print("Chronological split checks passed.")
print("Total forecasting windows:", len(windows))

In [0]:
print("Tracking URI:", mlflow.get_tracking_uri())

experiment = mlflow.get_experiment_by_name(
    config["mlflow"]["experiment_name"]
)

if experiment is not None:
    print("Experiment ID:", experiment.experiment_id)
    print("Experiment name:", experiment.name)

    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )

    display(runs[[
        "run_id",
        "tags.mlflow.runName",
        "metrics.validation_mae",
        "metrics.test_mae",
    ]])
else:
    print("Experiment not found.")